[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/06_Sensor_Noise_and_Failure.ipynb)

# DiveLab

## Notebook 06 — Sensor Noise, Estimation and Failure

**Guiding question:** What happens when the controller reacts to an imperfect measurement of depth?

*The controller never sees the true state directly — it sees a measurement.*

## Learning objectives

By the end of this lab, you will be able to:

- distinguish true state, measurement and estimate;
- model noisy depth measurements;
- simulate bias, drift, spikes, dropout and frozen-sensor failures;
- understand why differentiating noisy depth measurements produces noisy velocity estimates;
- apply a simple low-pass filter;
- explain the tradeoff between noise reduction and delay;
- discuss basic fault detection and plausibility checks.

## From Notebook 05 to Notebook 06

Notebook 05 showed that delayed feedback can degrade stability.

Now we introduce another limitation:

> the controller may react quickly, but to an imperfect measurement.

In control terminology, the true system state is:

$$
x(t)
$$

The sensor produces a measurement:

$$
y(t)
$$

and the controller may use an estimate:

$$
\hat{x}(t)
$$

These are not the same thing.

## State, measurement and estimate

For depth:

$$
z(t)
$$

is the true physical depth.

A depth sensor may provide:

$$
z_m(t)
$$

and after filtering or estimation we may use:

$$
\hat z(t)
$$

So:

$$
\boxed{
z(t)\neq z_m(t)\neq \hat z(t)
}
$$

in general.

## Measurement model

A simple sensor model is:

$$
z_m(t)=z(t)+n(t)
$$

where $n(t)$ is measurement noise.

A more general model is:

$$
z_m(t)=z(t)+b(t)+n(t)
$$

where:

- $n(t)$ is random noise;
- $b(t)$ is bias or drift.

## Types of sensor imperfection

In this notebook we will study:

1. random measurement noise;
2. constant bias;
3. slowly varying drift;
4. spike / outlier;
5. dropout;
6. frozen sensor;
7. filtering and estimation.

The last two cases are especially important because a sensor can fail while still returning apparently valid numbers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Reuse the buoyancy-control model

In [ ]:
rho = 1025.0
g = 9.80665
P0 = 101325.0

mass = 90.0
z_e = 20.0
v_e = 0.0

gas_surface_volume_e = 0.005

Cd = 0.9
A_drag = 0.7

Kz = 0.00008
Kv = 0.0008

In [ ]:
def pressure_at_depth(z):
    return P0 + rho * g * z

def gas_volume_at_depth(z, surface_volume):
    return surface_volume * P0 / pressure_at_depth(z)

gas_volume_e = gas_volume_at_depth(z_e, gas_surface_volume_e)
fixed_volume = mass / rho - gas_volume_e

def buoyant_force(z, surface_gas_volume):
    Vg = gas_volume_at_depth(z, surface_gas_volume)
    return rho * g * (fixed_volume + Vg)

def drag_force(v):
    return 0.5 * rho * Cd * A_drag * v * abs(v)

def acceleration(z, v, surface_gas_volume):
    return (
        buoyant_force(z, surface_gas_volume)
        - mass * g
        - drag_force(v)
    ) / mass

## Ideal controller

With perfect state information:

$$
u=K_z(z-z_e)-K_vv
$$

But a real controller usually does not know $z$ and $v$ exactly.

It receives sensor data instead.

In [ ]:
def ideal_controller(z, v):
    return Kz * (z - z_e) - Kv * v

## Build a reference trajectory

First simulate a controlled motion using the true state.

This gives us a reference against which sensor imperfections can be compared.

In [ ]:
def simulate_true_state(
    duration=30.0,
    dt=0.01,
    z0=z_e,
    v0=0.05,
    Vs0=gas_surface_volume_e,
    u_limit=0.0005
):
    n = int(duration / dt) + 1
    t = np.linspace(0, duration, n)

    z = np.zeros(n)
    v = np.zeros(n)
    Vs = np.zeros(n)
    u_hist = np.zeros(n)

    z[0] = z0
    v[0] = v0
    Vs[0] = Vs0

    for i in range(n - 1):
        u = ideal_controller(z[i], v[i])
        u = np.clip(u, -u_limit, u_limit)

        a = acceleration(z[i], v[i], Vs[i])

        v[i + 1] = v[i] + a * dt
        z[i + 1] = z[i] - v[i + 1] * dt
        Vs[i + 1] = max(Vs[i] + u * dt, 0.0)

        u_hist[i] = u

    u_hist[-1] = u_hist[-2]
    return t, z, v, Vs, u_hist

In [ ]:
t_ref, z_ref, v_ref, Vs_ref, u_ref = simulate_true_state()

# Part 1 — Random measurement noise

Suppose the depth sensor is accurate on average but noisy:

$$
z_m=z+n
$$

with:

$$
n\sim\mathcal{N}(0,\sigma^2)
$$

In [ ]:
rng = np.random.default_rng(42)

sigma_z = 0.08  # depth noise standard deviation [m]
noise = rng.normal(0.0, sigma_z, size=len(z_ref))

z_meas = z_ref + noise

In [ ]:
plt.plot(t_ref, z_ref, label="True depth")
plt.plot(t_ref, z_meas, alpha=0.6, label="Measured depth")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("True depth vs noisy sensor measurement")
plt.grid(True)
plt.legend()
plt.show()

The sensor still follows the true depth, but each sample fluctuates.

A controller that reacts too strongly to every fluctuation may produce unnecessary control activity.

# Part 2 — Estimating velocity from noisy depth

Suppose we do not have a direct vertical-velocity sensor.

A simple estimate is:

$$
\hat v \approx -\frac{\Delta z_m}{\Delta t}
$$

because positive upward velocity means decreasing depth.

In [ ]:
dt = t_ref[1] - t_ref[0]

v_est_raw = -np.gradient(z_meas, dt)

In [ ]:
plt.plot(t_ref, v_ref, label="True velocity")
plt.plot(t_ref, v_est_raw, alpha=0.6, label="Velocity from noisy depth")

plt.xlabel("Time [s]")
plt.ylabel("Upward velocity [m/s]")
plt.title("Numerical differentiation amplifies measurement noise")
plt.grid(True)
plt.legend()
plt.show()

## Why differentiation amplifies noise

The derivative estimates changes between nearby samples.

Small random variations in depth can therefore become large variations after division by the small timestep:

$$
\frac{\Delta n}{\Delta t}
$$

This is a fundamental signal-processing problem:

> differentiation emphasizes high-frequency noise.

# Part 3 — Low-pass filtering

A simple exponential low-pass filter is:

$$
\hat z_k
=
\alpha z_{m,k}
+
(1-\alpha)\hat z_{k-1}
$$

with:

$$
0<\alpha\le1
$$

Smaller $\alpha$ gives more smoothing.

In [ ]:
def low_pass_filter(signal, alpha):
    filtered = np.zeros_like(signal)
    filtered[0] = signal[0]

    for k in range(1, len(signal)):
        filtered[k] = alpha * signal[k] + (1 - alpha) * filtered[k - 1]

    return filtered

In [ ]:
alpha = 0.08
z_filt = low_pass_filter(z_meas, alpha)

In [ ]:
plt.plot(t_ref, z_ref, label="True depth")
plt.plot(t_ref, z_meas, alpha=0.35, label="Measured depth")
plt.plot(t_ref, z_filt, label="Filtered depth")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Low-pass filtering of depth measurement")
plt.grid(True)
plt.legend()
plt.show()

## Filtering reduces noise — but introduces lag

The filtered signal is smoother.

However, it reacts more slowly to real changes.

This creates a direct connection with Notebook 05:

> more filtering → less noise, but more effective delay.

So filtering itself creates a control-design tradeoff.

In [ ]:
v_est_filt = -np.gradient(z_filt, dt)

plt.plot(t_ref, v_ref, label="True velocity")
plt.plot(t_ref, v_est_raw, alpha=0.3, label="Raw derivative")
plt.plot(t_ref, v_est_filt, label="Derivative after filtering")

plt.xlabel("Time [s]")
plt.ylabel("Upward velocity [m/s]")
plt.title("Filtering before differentiation")
plt.grid(True)
plt.legend()
plt.show()

# Part 4 — Constant sensor bias

Now suppose the sensor is consistently wrong by a fixed amount:

$$
z_m=z+b
$$

For example:

$$
b=+0.5\ \mathrm{m}
$$

In [ ]:
bias = 0.5
z_bias = z_ref + bias

In [ ]:
plt.plot(t_ref, z_ref, label="True depth")
plt.plot(t_ref, z_bias, label="Biased measurement")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Constant sensor bias")
plt.grid(True)
plt.legend()
plt.show()

A biased sensor can be dangerous from a control perspective because the reading may look smooth and plausible.

The controller may stabilize the measured depth while the true depth remains offset.

A stable closed loop can therefore still be systematically wrong.

# Part 5 — Sensor drift

A drifting sensor has a bias that changes slowly over time:

$$
z_m(t)=z(t)+b(t)
$$

In [ ]:
drift_rate = 0.02  # m per second, intentionally exaggerated for demonstration
drift = drift_rate * t_ref

z_drift = z_ref + drift

In [ ]:
plt.plot(t_ref, z_ref, label="True depth")
plt.plot(t_ref, z_drift, label="Drifting measurement")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Slow sensor drift")
plt.grid(True)
plt.legend()
plt.show()

Drift is harder to detect than random noise because each individual measurement may appear reasonable.

The error emerges gradually.

# Part 6 — Spike / outlier

A sensor may occasionally produce a single implausible reading.

For example:

$$
20.0,\ 20.1,\ 20.0,\ \boxed{13.5},\ 19.9,\ 20.0
$$

In [ ]:
z_spike = z_ref.copy()

spike_time = 12.0
spike_idx = np.argmin(np.abs(t_ref - spike_time))
z_spike[spike_idx] -= 6.0

In [ ]:
plt.plot(t_ref, z_ref, label="True depth")
plt.plot(t_ref, z_spike, label="Sensor with spike")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Single measurement outlier")
plt.grid(True)
plt.legend()
plt.show()

A controller that trusts this single value blindly may command a very large correction.

This motivates:

- plausibility limits;
- rate-of-change limits;
- median filters;
- redundant sensors;
- fault detection.

# Part 7 — Sensor dropout

A dropout means that no new measurement is available.

We represent missing values with:

```python
np.nan
```

In [ ]:
z_dropout = z_ref.copy()

drop_start = 10.0
drop_end = 14.0

mask = (t_ref >= drop_start) & (t_ref <= drop_end)
z_dropout[mask] = np.nan

In [ ]:
plt.plot(t_ref, z_ref, label="True depth")
plt.plot(t_ref, z_dropout, label="Sensor measurement")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Depth-sensor dropout")
plt.grid(True)
plt.legend()
plt.show()

## What should the controller do during dropout?

Possible strategies include:

- hold the last valid measurement;
- rely on another sensor;
- propagate a model-based estimate;
- enter a degraded or fail-safe mode.

Simply pretending the measurement still exists is not a robust strategy.

In [ ]:
def hold_last_valid(signal):
    held = signal.copy()

    for i in range(1, len(held)):
        if np.isnan(held[i]):
            held[i] = held[i - 1]

    return held

z_dropout_held = hold_last_valid(z_dropout)

In [ ]:
plt.plot(t_ref, z_ref, label="True depth")
plt.plot(t_ref, z_dropout_held, label="Last value held")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Dropout handled by holding last valid measurement")
plt.grid(True)
plt.legend()
plt.show()

Holding the last value avoids missing data, but creates another problem:

> the controller now acts on an increasingly old measurement.

So a sensor dropout can effectively become a **growing delay**.

# Part 8 — Frozen sensor

A frozen sensor is more subtle.

After some failure time $t_f$:

$$
z_m(t)=z_m(t_f)
$$

even while the true depth changes.

In [ ]:
z_frozen = z_ref.copy()

freeze_time = 10.0
freeze_idx = np.argmin(np.abs(t_ref - freeze_time))

z_frozen[freeze_idx:] = z_frozen[freeze_idx]

In [ ]:
plt.plot(t_ref, z_ref, label="True depth")
plt.plot(t_ref, z_frozen, label="Frozen sensor")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Frozen depth sensor")
plt.grid(True)
plt.legend()
plt.show()

A frozen sensor may be harder to detect than a dropout because it continues to return apparently valid numeric values.

This motivates consistency checks such as:

- "Has the measurement changed at all?";
- "Is it compatible with velocity or acceleration information?";
- "Does it agree with redundant sensors?";
- "Does the model predict something different?"

# Part 9 — A controller using measured depth

Now let us close the loop using an imperfect depth measurement.

To keep the example simple, we estimate velocity from the filtered measured depth.

In [ ]:
def simulate_with_sensor(
    noise_std=0.05,
    bias=0.0,
    alpha=0.08,
    duration=30.0,
    dt=0.01,
    z0=z_e,
    v0=0.05,
    Vs0=gas_surface_volume_e,
    u_limit=0.0005,
    seed=1,
):
    rng = np.random.default_rng(seed)

    n = int(duration / dt) + 1
    t = np.linspace(0, duration, n)

    z = np.zeros(n)
    v = np.zeros(n)
    Vs = np.zeros(n)

    z_meas = np.zeros(n)
    z_hat = np.zeros(n)
    v_hat = np.zeros(n)
    u_hist = np.zeros(n)

    z[0] = z0
    v[0] = v0
    Vs[0] = Vs0

    z_meas[0] = z0 + bias + rng.normal(0.0, noise_std)
    z_hat[0] = z_meas[0]
    v_hat[0] = 0.0

    for i in range(n - 1):

        u = Kz * (z_hat[i] - z_e) - Kv * v_hat[i]
        u = np.clip(u, -u_limit, u_limit)

        a = acceleration(z[i], v[i], Vs[i])

        v[i + 1] = v[i] + a * dt
        z[i + 1] = z[i] - v[i + 1] * dt
        Vs[i + 1] = max(Vs[i] + u * dt, 0.0)

        z_meas[i + 1] = (
            z[i + 1]
            + bias
            + rng.normal(0.0, noise_std)
        )

        z_hat[i + 1] = (
            alpha * z_meas[i + 1]
            + (1 - alpha) * z_hat[i]
        )

        v_hat[i + 1] = -(z_hat[i + 1] - z_hat[i]) / dt

        u_hist[i] = u

    u_hist[-1] = u_hist[-2]

    return t, z, v, Vs, z_meas, z_hat, v_hat, u_hist

## Compare true and estimated state

In [ ]:
t_s, z_s, v_s, Vs_s, zm_s, zh_s, vh_s, u_s = simulate_with_sensor()

In [ ]:
plt.plot(t_s, z_s, label="True depth")
plt.plot(t_s, zm_s, alpha=0.3, label="Measured depth")
plt.plot(t_s, zh_s, label="Estimated depth")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("True, measured and estimated depth")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.plot(t_s, v_s, label="True velocity")
plt.plot(t_s, vh_s, label="Estimated velocity")

plt.xlabel("Time [s]")
plt.ylabel("Upward velocity [m/s]")
plt.title("True vs estimated vertical velocity")
plt.grid(True)
plt.legend()
plt.show()

## The estimator is part of the loop

We now have:

> plant → sensor → estimator → controller → actuator → plant

The controller no longer uses the true state.

It uses:

$$
\hat x
$$

This is the beginning of **output-feedback control**.

# Part 10 — Noise-filtering tradeoff

Let's compare several filter strengths.

In [ ]:
alphas = [0.02, 0.08, 0.3]

for a in alphas:
    t_a, z_a, v_a, Vs_a, zm_a, zh_a, vh_a, u_a = simulate_with_sensor(alpha=a)
    plt.plot(t_a, zh_a, label=f"alpha = {a}")

plt.plot(t_s, z_s, "--", label="True depth")

plt.xlabel("Time [s]")
plt.ylabel("Depth estimate [m]")
plt.title("Filtering tradeoff")
plt.grid(True)
plt.legend()
plt.show()

Smaller $\alpha$:

- suppresses more noise;
- reacts more slowly.

Larger $\alpha$:

- follows changes more quickly;
- lets more noise through.

This is a classic estimation tradeoff:

> noise rejection vs responsiveness.

# Part 11 — Simple plausibility checks

One basic fault-detection idea is to reject measurements that change faster than physically plausible.

For example:

$$
|z_{m,k}-z_{m,k-1}|>\Delta z_{\max}
$$

may indicate a spike.

In [ ]:
def reject_large_jumps(signal, max_jump):
    cleaned = signal.copy()

    for i in range(1, len(cleaned)):
        if abs(cleaned[i] - cleaned[i - 1]) > max_jump:
            cleaned[i] = cleaned[i - 1]

    return cleaned

In [ ]:
z_spike_clean = reject_large_jumps(z_spike, max_jump=0.5)

plt.plot(t_ref, z_ref, label="True depth")
plt.plot(t_ref, z_spike, alpha=0.4, label="With spike")
plt.plot(t_ref, z_spike_clean, label="After plausibility check")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Simple outlier rejection")
plt.grid(True)
plt.legend()
plt.show()

This is intentionally simple.

Real fault detection can use:

- redundant sensors;
- residuals between model and measurement;
- statistical thresholds;
- consistency checks;
- observers and Kalman filters;
- fault-tolerant control logic.

## Systems-theory interpretation

A general linear measurement model is:

$$
y=Cx+n
$$

where:

- $x$ is the state;
- $y$ is the measured output;
- $C$ maps the state to measured quantities;
- $n$ is measurement noise.

If not every state is measured directly, an estimator may reconstruct:

$$
\hat x
$$

from:

- measurements;
- inputs;
- a model of the plant.

## Observability preview

A natural question now appears:

> Can the internal state be reconstructed from the measured output?

That is the concept of **observability**.

For example, if we measure depth but not velocity, can we infer velocity from the time history of depth and the system model?

Notebook 07 can develop this idea formally.

## Exercises

### 1. Increase depth noise

Try:

```python
noise_std = 0.02
noise_std = 0.10
noise_std = 0.30
```

Observe the effect on:

- measured depth;
- velocity estimate;
- control effort.

### 2. Change filter strength

Try:

```python
alpha = 0.02
alpha = 0.10
alpha = 0.50
```

Compare smoothness and lag.

### 3. Add a constant bias

Run the sensor-controlled simulation with:

```python
bias = 0.5
```

Does the controller stabilize the true depth at $z_e$?

### 4. Frozen sensor

Modify the simulation so that the measurement freezes after 10 s.

What happens to the controller?

### 5. Dropout

Replace the depth measurement with `np.nan` for a few seconds.

Try two strategies:

- hold the last measurement;
- continue using a model-based prediction.

## Challenge — residual-based fault detection

Suppose a model predicts:

$$
\hat z_{k|k-1}
$$

before the next measurement arrives.

Define the residual:

$$
r_k=z_{m,k}-\hat z_{k|k-1}
$$

Use the residual to detect:

- a large spike;
- a slowly growing bias;
- a frozen sensor.

Which failure is easiest to detect?

In [ ]:
# Your code here

## Summary

In this lab we learned that:

- the true state is not the same as the measured output;
- sensor noise affects feedback performance;
- numerical differentiation amplifies noise;
- filtering suppresses noise but introduces lag;
- bias can produce systematic control error;
- drift can develop slowly and remain plausible;
- dropout creates missing information;
- a frozen sensor can be more deceptive than a dropout;
- spikes motivate plausibility checks and robust estimation;
- estimation is part of the feedback loop;
- observability determines whether hidden states can be reconstructed.

### Core insight

> **A controller is only as good as the information it receives — and as the estimator that interprets it.**

### Next

Notebook 07 can introduce **observability and state estimation**, leading naturally toward observers and the Kalman filter.